# Full Spectrum Player Value (FSPV) — Interactive Demo
### SkillCorner Basketball Analytics Cup (Liga ACB 2025/2026)

This interactive notebook demonstrates the end-to-end execution of the **Full Spectrum Player Value** pipeline on a sample game (`114243` — BAXI Manresa vs. Coviran Granada):
1. **Event Ingestion & Action Sequencing (BAV)**
2. **State-Space Scoring Inference (XGBoost)**
3. **Spatial Graph Construction & GNN Evaluation (SCI)**
4. **Full Spectrum Synthesis & Visualization**


In [ ]:
# Setup dependencies and import use cases
import sys
sys.path.append('src')

from infrastructure.event_loader import load_events, load_player_directory
from use_cases.bav.action_sequencer import ActionSequencer
from use_cases.bav.state_extractor import StateExtractor
from use_cases.bav.bav_model import BAVModel
from use_cases.bav.bav_scorer import BAVScorer
from use_cases.player_value.combiner import PlayerValueCombiner
from use_cases.bav.possession_value_map import PossessionValueMap

print('Environment ready!')

In [ ]:
# 1. Load sample game events and sequence actions
events = load_events(114243)
sequencer = ActionSequencer()
actions = sequencer.extract_sequences(events, game_id=114243)
print(f'Extracted {len(actions)} sequenced actions for Game 114243.')
actions.head(5)

In [ ]:
# 2. Extract features and compute possession value map
extractor = StateExtractor()
feats = extractor.extract_features(actions, events)

# Score with pretrained model
model = BAVModel()
model.load_model('models/bav_xgboost.json')
scorer = BAVScorer(model=model)
scored_actions = scorer.score_actions(feats)

val_map = PossessionValueMap()
grid_res = val_map.compute_grid(scored_actions)
print('Top 3 high-value court hotspots:')
for h in grid_res['hotspots'][:3]:
    print(f"  Location: ({h['x_meters']}m, {h['y_meters']}m) -> Expected BAV: {h['bav_expected']:+.4f}")

In [ ]:
# 3. Combine scores and display top performers
player_dir = load_player_directory()
combiner = PlayerValueCombiner()
rankings = combiner.combine(
    bav_scores='outputs/scores/bav_scores.parquet',
    sci_scores='outputs/scores/sci_scores.parquet',
    player_metadata=player_dir,
)
print('Top 5 Performers in Full Spectrum Player Value:')
rankings.head(5)